In [0]:

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    to_date
)

# Existing Silver Customers table
silver_customers_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/customers"
)

# New SCD Type 2 target table
scd_customer_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "gold/dim_customer"
)

# Read cleaned customer data from Silver
silver_customers_df = (
    spark.read
    .format("delta")
    .load(silver_customers_path)
)

print("Silver customer count:", silver_customers_df.count())

display(silver_customers_df.limit(5))


Silver customer count: 100


customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
12,Akash,Reddy,akash.reddy12@example.com,Jaipur,Rajasthan,India,Consumer,2024-03-22,2024-05-20
18,Vivaan,Reddy,vivaan.reddy18@example.com,Chennai,Tamil Nadu,India,Consumer,2025-02-14,2025-04-23
38,Priya,Kulkarni,priya.kulkarni38@example.com,Ahmedabad,Gujarat,India,Premium,2025-01-17,2025-05-09
67,Isha,Joshi,isha.joshi67@example.com,Nagpur,Maharashtra,India,Premium,2025-03-24,2025-08-10
70,Arjun,Pawar,arjun.pawar70@example.com,Mumbai,Maharashtra,India,Corporate,2025-07-20,2025-08-28


In [0]:

scd_customer_initial_df = (
    silver_customers_df
    .withColumn(
        "effective_start_date",
        current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        to_date(lit("9999-12-31"))
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

(
    scd_customer_initial_df.write
    .format("delta")
    .mode("overwrite")
    .save(scd_customer_path)
)

print("Initial SCD Type 2 customer dimension created.")

print(
    "Dimension row count:",
    spark.read.format("delta").load(scd_customer_path).count()
)

display(
    spark.read
    .format("delta")
    .load(scd_customer_path)
    .select(
        "customer_id",
        "customer_segment",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .limit(10)
)


Initial SCD Type 2 customer dimension created.
Dimension row count: 100


customer_id,customer_segment,effective_start_date,effective_end_date,is_current
12,Consumer,2026-10-10T21:18:21.730Z,9999-12-31,true
18,Consumer,2026-10-10T21:18:21.730Z,9999-12-31,true
38,Premium,2026-10-10T21:18:21.730Z,9999-12-31,true
67,Premium,2026-10-10T21:18:21.730Z,9999-12-31,true
70,Corporate,2026-10-10T21:18:21.730Z,9999-12-31,true
93,Premium,2026-10-10T21:18:21.730Z,9999-12-31,true
16,Premium,2026-10-10T21:18:21.730Z,9999-12-31,true
64,Corporate,2026-10-10T21:18:21.730Z,9999-12-31,true
74,Premium,2026-10-10T21:18:21.730Z,9999-12-31,true
94,Premium,2026-10-10T21:18:21.730Z,9999-12-31,true


In [0]:

from pyspark.sql.functions import (
    col,
    when,
    lit,
    current_timestamp
)

# Read the current SCD dimension
scd_customer_df = (
    spark.read
    .format("delta")
    .load(scd_customer_path)
)

# Find customer 1's current segment
customer_before_df = (
    scd_customer_df
    .filter(
        (col("customer_id") == 1) &
        (col("is_current") == True)
    )
)

display(
    customer_before_df.select(
        "customer_id",
        "customer_segment",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
)

# Simulate a change to a different segment
current_segment = (
    customer_before_df
    .select("customer_segment")
    .first()["customer_segment"]
)

new_segment = (
    "Corporate"
    if current_segment == "Premium"
    else "Premium"
)

print("Old segment:", current_segment)
print("New segment:", new_segment)


customer_id,customer_segment,effective_start_date,effective_end_date,is_current
1,Consumer,2026-10-10T21:18:21.730Z,9999-12-31,true


Old segment: Consumer
New segment: Premium


In [0]:

# Get the customer's existing attributes
customer_source_df = (
    silver_customers_df
    .filter(col("customer_id") == 1)
    .withColumn(
        "customer_segment",
        when(
            col("customer_segment") == current_segment,
            lit(new_segment)
        ).otherwise(col("customer_segment"))
    )
    .withColumn(
        "effective_start_date",
        current_timestamp()
    )
    .withColumn(
        "effective_end_date",
        to_date(lit("9999-12-31"))
    )
    .withColumn("is_current", lit(True))
)

display(
    customer_source_df.select(
        "customer_id",
        "customer_segment",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
)


customer_id,customer_segment,effective_start_date,effective_end_date,is_current
1,Premium,2026-10-10T21:19:20.622Z,9999-12-31,true


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col,
    lit,
    current_date,
    sha2,
    concat_ws,
    coalesce
)

# ============================================================
# STEP 1: LOAD THE EXISTING SCD TYPE 2 DELTA TABLE
# ============================================================

scd_delta = DeltaTable.forPath(
    spark,
    scd_customer_path
)

# ============================================================
# STEP 2: IDENTIFY NEW AND CHANGED CUSTOMERS
# ============================================================

# Compare business attributes, not SCD tracking columns.
business_columns = [
    "first_name",
    "last_name",
    "email",
    "city",
    "state",
    "country",
    "customer_segment"
]

source_df = customer_source_df.dropDuplicates(["customer_id"])

target_current_df = (
    scd_delta.toDF()
    .filter(col("is_current") == True)
    .select(
        "customer_id",
        *business_columns
    )
)

# Hash the business attributes to detect changes.
source_hashed_df = source_df.withColumn(
    "source_hash",
    sha2(
        concat_ws(
            "||",
            *[
                coalesce(col(c).cast("string"), lit("<NULL>"))
                for c in business_columns
            ]
        ),
        256
    )
)

target_hashed_df = target_current_df.withColumn(
    "target_hash",
    sha2(
        concat_ws(
            "||",
            *[
                coalesce(col(c).cast("string"), lit("<NULL>"))
                for c in business_columns
            ]
        ),
        256
    )
)

comparison_df = (
    source_hashed_df.alias("source")
    .join(
        target_hashed_df.alias("target"),
        on="customer_id",
        how="left"
    )
)

# New customers or customers with changed business attributes.
changes_df = comparison_df.filter(
    col("target_hash").isNull()
    | (col("source_hash") != col("target_hash"))
).select(
    *[col(f"source.{c}").alias(c) for c in source_df.columns]
)

# ============================================================
# STEP 3: STAGE EXPIRE AND INSERT ACTIONS
# ============================================================

# Only changed customers that already exist need expiration.
existing_ids_df = target_current_df.select("customer_id")

changed_existing_df = changes_df.join(
    existing_ids_df,
    on="customer_id",
    how="inner"
)

# EXPIRE rows need all source columns so UNION schemas match.
expire_rows = (
    changed_existing_df
    .withColumn("merge_key", col("customer_id"))
    .withColumn("change_action", lit("EXPIRE"))
)

# INSERT rows deliberately use a NULL merge key so they do not match.
insert_rows = (
    changes_df
    .withColumn("merge_key", lit(None).cast("int"))
    .withColumn("change_action", lit("INSERT"))
)

staged_changes_df = expire_rows.unionByName(
    insert_rows,
    allowMissingColumns=False
)

# ============================================================
# STEP 4: APPLY THE SCD TYPE 2 MERGE
# ============================================================

(
    scd_delta.alias("target")
    .merge(
        staged_changes_df.alias("source"),
        """
        target.customer_id = source.merge_key
        AND target.is_current = true
        """
    )
    .whenMatchedUpdate(
        condition="source.change_action = 'EXPIRE'",
        set={
            "is_current": "false",
            "effective_end_date": "current_date()"
        }
    )
    .whenNotMatchedInsert(
        condition="source.change_action = 'INSERT'",
        values={
            "customer_id": "source.customer_id",
            "first_name": "source.first_name",
            "last_name": "source.last_name",
            "email": "source.email",
            "city": "source.city",
            "state": "source.state",
            "country": "source.country",
            "customer_segment": "source.customer_segment",
            "created_date": "source.created_date",
            "updated_date": "source.updated_date",
            "effective_start_date": "current_date()",
            "effective_end_date": "CAST(NULL AS DATE)",
            "is_current": "true"
        }
    )
    .execute()
)

print("SCD Type 2 MERGE completed.")
print("Changed/new customers staged:", changes_df.count())

# ============================================================
# STEP 5: VALIDATE THE RESULT
# ============================================================

display(
    scd_delta.toDF()
    .groupBy("is_current")
    .count()
)

display(
    scd_delta.toDF()
    .orderBy("customer_id", "effective_start_date")
)


SCD Type 2 MERGE completed.
Changed/new customers staged: 0


is_current,count
true,100
false,1


customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date,effective_start_date,effective_end_date,is_current
1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Premium,2024-10-08,2024-12-09,2026-10-10T00:00:00.000Z,null,true
1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09,2026-10-10T21:18:21.730Z,2026-10-10,false
2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03,2026-10-10T21:18:21.730Z,9999-12-31,true
3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30,2026-10-10T21:18:21.730Z,9999-12-31,true
4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16,2026-10-10T21:18:21.730Z,9999-12-31,true
5,Riya,Kulkarni,riya.kulkarni5@example.com,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01,2026-10-10T21:18:21.730Z,9999-12-31,true
6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07,2026-10-10T21:18:21.730Z,9999-12-31,true
7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02,2026-10-10T21:18:21.730Z,9999-12-31,true
8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31,2026-10-10T21:18:21.730Z,9999-12-31,true
9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30,2026-10-10T21:18:21.730Z,9999-12-31,true


In [0]:

scd_result_df = (
    spark.read
    .format("delta")
    .load(scd_customer_path)
)

print("Total historical rows:", scd_result_df.count())

display(
    scd_result_df
    .filter(col("customer_id") == 1)
    .select(
        "customer_id",
        "customer_segment",
        "effective_start_date",
        "effective_end_date",
        "is_current"
    )
    .orderBy("effective_start_date")
)


Total historical rows: 101


customer_id,customer_segment,effective_start_date,effective_end_date,is_current
1,Premium,2026-10-10T00:00:00.000Z,null,true
1,Consumer,2026-10-10T21:18:21.730Z,2026-10-10,false
